# AutoSplice · CocoGlide JPEG 저장 이력 확인

- 작성자: seonghun
- 상태: **완료 — Colab 저장 출력 검토 완료 (2026-11-23), 학습·평가 조건은 팀 논의 필요**
- 작성일: 2026-11-23
- 관련 작업: T05 데이터셋 EDA·분할 확정
- [보고서](../../docs/eda/EXP_seonghun_003_jpeg_history_check.md) · [byhyun 001 보고서](../../docs/eda/EXP_byhyun_001_preprocessing_check.md)

byhyun 001은 train 표본 200장에서 정상 원본이 모두 JPEG 품질 75·4:2:0, 조작본(JPEG100 폴더)이 모두 품질 100·4:2:0으로 저장된 것을 확인함. 이 차이가 전체에서도 같다면 모델이 조작 흔적이 아니라 압축 품질 차이만으로 정상·조작을 구분할 수 있음. 이 노트북은 같은 차이가 train·val·test 전체와 CocoGlide에서도 나타나는지 확인함.

파일 헤더의 양자화 테이블과 서브샘플링을 읽어 저장 품질을 추정함. 이미지 픽셀은 디코드하지 않음. 데이터·분할·학습 설정을 바꾸지 않으며 test와 CocoGlide는 저장 형식 확인에만 사용함.

팀원이 VS Code의 Colab 원격 커널에서 위에서부터 실행하고 출력이 포함된 노트북을 저장함. GPU는 필요 없음.

## 1. Drive·공통 코드·실행 환경 준비

공유 Drive의 데이터 ZIP과 `AI_Forensics/project/scripts/`를 사용함. 실제 import 위치와 코드·분할 해시를 확인함. 다르면 중단하고 버전을 확인해야 하며 기대 해시만 바꾸지 않음.

In [9]:
from pathlib import Path
from google.colab import drive
import importlib
import importlib.metadata
import platform
import sys
from datetime import datetime, timezone
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image, JpegImagePlugin, features
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
drive.mount("/content/drive")
shared_root = Path("/content/drive/MyDrive/AI_Forensics")
project_root = shared_root / "project"
work_root = Path("/content")
data_root = work_root / "datasets"
split_path = project_root / "scripts/splits/autosplice_seed42.csv"
archive_paths = {
    "AutoSplice": shared_root / "datasets/AutoSplice/AutoSplice.zip",
    "CocoGlide": shared_root / "datasets/CocoGlide.zip",
}
extract_paths = {"AutoSplice": data_root, "CocoGlide": data_root / "CocoGlide"}
required = ["scripts/__init__.py", "scripts/datasets.py", "scripts/preprocessing.py", "scripts/splits/autosplice_seed42.csv"]
for relative in required:
    if not (project_root / relative).is_file():
        raise FileNotFoundError(f"Drive에 공통 파일을 업로드해야 함: {relative}")

sys.path.insert(0, str(project_root))
importlib.invalidate_caches()
import scripts
import scripts.datasets as dataset_module

for module, relative in [(scripts, "scripts/__init__.py"), (dataset_module, "scripts/datasets.py")]:
    if Path(module.__file__).resolve() != (project_root / relative).resolve():
        raise ImportError(f"프로젝트의 모듈을 불러오도록 커널을 재시작해야 함: {relative}")
    print("불러온 코드:", module.__file__)

from scripts.datasets import sha256, prepare_dataset_zip, autosplice_records, cocoglide_records

print("실행 시각 UTC:", datetime.now(timezone.utc).isoformat())
print("Python:", platform.python_version())
print("OS:", platform.platform())
for package in ("numpy", "Pillow", "pandas"):
    print(package, importlib.metadata.version(package))
print("JPEG library:", features.version("jpg"))
print("GPU: 사용하지 않음")
print("프로젝트:", project_root)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
불러온 코드: /content/drive/MyDrive/AI_Forensics/project/scripts/__init__.py
불러온 코드: /content/drive/MyDrive/AI_Forensics/project/scripts/datasets.py
실행 시각 UTC: 2026-11-23T12:02:50.062670+00:00
Python: 3.13.16
OS: Linux-6.6.159+-x86_64-with-glibc2.39
numpy 2.4.6
Pillow 11.3.0
pandas 2.2.3
JPEG library: 6.2
GPU: 사용하지 않음
프로젝트: /content/drive/MyDrive/AI_Forensics/project


In [10]:
expected_files = {
    "scripts/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "scripts/datasets.py": "435215ef08c2c43ce2f84795d783895074eee85f8e062f28ea8ee6acabad3e88",
    "scripts/preprocessing.py": "73ce7fa0aee3f2b901a9ab04d430ff8ae2bcaa7262005600f6b08f04b95c652f",
    "scripts/splits/autosplice_seed42.csv": "f8a2893ab93b82264e6434fa0c26f09472de05bd98829065c2a0db846ace7b1e",
}
for relative, expected_hash in expected_files.items():
    actual_hash = sha256(project_root / relative)
    print(relative, actual_hash)
    if actual_hash != expected_hash:
        raise ValueError(f"검토한 코드·분할 버전과 일치해야 함: {relative}")

scripts/__init__.py e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
scripts/datasets.py 435215ef08c2c43ce2f84795d783895074eee85f8e062f28ea8ee6acabad3e88
scripts/preprocessing.py 73ce7fa0aee3f2b901a9ab04d430ff8ae2bcaa7262005600f6b08f04b95c652f
scripts/splits/autosplice_seed42.csv f8a2893ab93b82264e6434fa0c26f09472de05bd98829065c2a0db846ace7b1e


## 2. 데이터 ZIP 준비

001·002와 같은 ZIP·공통 함수를 사용함. 기존 완료 표시가 있으면 해제 파일을 재사용하며 그 여부를 출력으로 남김.

In [11]:
archive_hashes = {
    "AutoSplice": "0d94e2df6842f44c11e21ee207707c49fd9717ec89048bd6f28fc1e07c4a5e36",
    "CocoGlide": "fc848643ad0a81010ba2a6ab8c055762782f54e4fb9e407210dbfe1c3b538af9",
}
for name, source in archive_paths.items():
    prepare_dataset_zip(source, work_root / f"{name}.zip", extract_paths[name], archive_hashes[name])
    print(name, "ZIP SHA-256:", archive_hashes[name])

AutoSplice 기존 해제 파일 사용
AutoSplice ZIP SHA-256: 0d94e2df6842f44c11e21ee207707c49fd9717ec89048bd6f28fc1e07c4a5e36
CocoGlide 기존 해제 파일 사용
CocoGlide ZIP SHA-256: fc848643ad0a81010ba2a6ab8c055762782f54e4fb9e407210dbfe1c3b538af9


## 3. 대상 파일 목록

- AutoSplice 정상 원본(`Authentic`)은 한 번만 등록함.
- AutoSplice 조작본은 `Forged_JPEG100`·`Forged_JPEG90`·`Forged_JPEG75` 세 폴더를 모두 등록함. 폴더 이름의 품질과 실제 저장 품질이 같은지도 확인함.
- CocoGlide는 정상(`real`)·조작(`fake`)을 모두 등록함.

split은 고정 분할 CSV를 따름.

In [12]:
rows = []
for quality in (100, 90, 75):
    for row in autosplice_records(data_root / "AutoSplice", split_path, quality=quality):
        if row["kind"] == "normal" and quality != 100:
            continue
        folder = "Authentic" if row["kind"] == "normal" else f"Forged_JPEG{quality}"
        rows.append(dict(dataset="AutoSplice", split=row["split"], kind=row["kind"], folder=folder, path=row["image_path"]))
for row in cocoglide_records(data_root / "CocoGlide"):
    rows.append(dict(dataset="CocoGlide", split=row["split"], kind=row["kind"], folder=row["image_path"].parent.name, path=row["image_path"]))
files = pd.DataFrame(rows)
if files["path"].duplicated().any():
    raise ValueError("대상 파일 경로는 중복되지 않아야 함.")
display(files.groupby(["dataset", "folder", "split"]).size().unstack(fill_value=0))

split                      external_test  test  train  val
dataset    folder                                         
AutoSplice Authentic                   0   228   1818  227
           Forged_JPEG100              0   335   2925  361
           Forged_JPEG75               0   335   2925  361
           Forged_JPEG90               0   335   2925  361
CocoGlide  fake                      512     0      0    0
           real                      512     0      0    0

## 4. 저장 품질 추정

JPEG 파일은 압축할 때 쓴 양자화 테이블을 헤더에 저장함. 표준(IJG) 테이블을 품질 1~100으로 조정해 쓰는 일반적인 인코더라면 테이블만으로 저장 품질을 알 수 있음.

기준 테이블은 이 런타임의 Pillow로 작은 이미지를 품질 1~100으로 직접 저장해 만듦. 같은 방식으로 읽기 때문에 테이블 순서 차이가 생기지 않음. 파일 테이블이 기준과 정확히 같으면 그 품질로, 아니면 가장 가까운 품질(테이블 값 차이의 합이 가장 작은 품질)로 표시하고 `exact=False`로 구분함. 서브샘플링은 0=4:4:4, 1=4:2:2, 2=4:2:0이며 파일에서 읽음. PNG 등 JPEG가 아닌 파일은 형식만 기록함.

In [13]:
def read_tables(image):
    tables = image.quantization
    return tuple(tables[0]), tuple(tables[1]) if 1 in tables else ()


reference = {}
sample = Image.fromarray(np.random.default_rng(0).integers(0, 256, (16, 16, 3), dtype=np.uint8))
for quality in range(1, 101):
    buffer = BytesIO()
    sample.save(buffer, format="JPEG", quality=quality)
    with Image.open(BytesIO(buffer.getvalue())) as image:
        reference[quality] = read_tables(image)
reference_array = {q: np.array(lum + chroma, dtype=np.int64) for q, (lum, chroma) in reference.items()}


def inspect(path):
    with Image.open(path) as image:
        result = dict(format=image.format, mode=image.mode, progressive=bool(image.info.get("progressive", False)))
        if image.format != "JPEG":
            return dict(result, quality=None, exact=None, subsampling=None, tables=None)
        lum, chroma = read_tables(image)
        result.update(subsampling=JpegImagePlugin.get_sampling(image), tables=len(image.quantization))
    values = np.array(lum + chroma, dtype=np.int64)
    exact = [q for q, (ref_lum, ref_chroma) in reference.items() if ref_lum == lum and ref_chroma == chroma]
    if exact:
        return dict(result, quality=exact[-1], exact=True)
    same_length = {q: ref for q, ref in reference_array.items() if len(ref) == len(values)}
    nearest = min(same_length, key=lambda q: np.abs(same_length[q] - values).sum()) if same_length else None
    return dict(result, quality=nearest, exact=False)


records, errors = [], []
for row in files.itertuples():
    try:
        records.append(dict(row._asdict(), **inspect(row.path)))
    except Exception as error:
        errors.append(dict(path=str(row.path), error=repr(error)))
print("확인한 파일:", len(records), "/ 오류:", len(errors))
if errors:
    display(pd.DataFrame(errors).head(20))
    raise RuntimeError("읽기 오류를 확인한 뒤 다음 단계로 진행해야 함.")
history = pd.DataFrame(records).drop(columns=["Index", "path"]).assign(path=files["path"].values)
history["quality"] = history["quality"].astype("Int64")

확인한 파일: 14160 / 오류: 0


## 5. 집계

그룹(데이터셋·폴더·split)별로 파일 형식, 추정 품질, 서브샘플링, 표준 테이블 일치 여부를 집계함. 정상과 조작의 품질이 모든 split에서 다르게 고정돼 있는지, JPEG90·75 폴더의 조작본이 폴더 이름의 품질로 저장돼 있는지 확인함.

In [14]:
keys = ["dataset", "folder", "split"]
print("파일 형식")
display(history.groupby(keys)["format"].value_counts().unstack(fill_value=0))

jpeg = history[history["format"] == "JPEG"]
print("추정 품질 (JPEG 파일)")
display(jpeg.groupby(keys)["quality"].value_counts().unstack(fill_value=0))
print("서브샘플링 (0=4:4:4, 1=4:2:2, 2=4:2:0)")
display(jpeg.groupby(keys)["subsampling"].value_counts().unstack(fill_value=0))
print("표준 테이블 정확 일치 비율·progressive 비율")
display(jpeg.groupby(keys).agg(files=("exact", "size"), exact_ratio=("exact", "mean"), progressive_ratio=("progressive", "mean")).round(4))

파일 형식


format                                   JPEG  PNG
dataset    folder         split                   
AutoSplice Authentic      test            228    0
                          train          1818    0
                          val             227    0
           Forged_JPEG100 test            335    0
                          train          2925    0
                          val             361    0
           Forged_JPEG75  test            335    0
                          train          2925    0
                          val             361    0
           Forged_JPEG90  test            335    0
                          train          2925    0
                          val             361    0
CocoGlide  fake           external_test     0  512
           real           external_test     0  512

추정 품질 (JPEG 파일)


quality                           75    90    100
dataset    folder         split                  
AutoSplice Authentic      test    228     0     0
                          train  1818     0     0
                          val     227     0     0
           Forged_JPEG100 test      0     0   335
                          train     0     0  2925
                          val       0     0   361
           Forged_JPEG75  test    335     0     0
                          train  2925     0     0
                          val     361     0     0
           Forged_JPEG90  test      0   335     0
                          train     0  2925     0
                          val       0   361     0

서브샘플링 (0=4:4:4, 1=4:2:2, 2=4:2:0)


subsampling                       2.0
dataset    folder         split      
AutoSplice Authentic      test    228
                          train  1818
                          val     227
           Forged_JPEG100 test    335
                          train  2925
                          val     361
           Forged_JPEG75  test    335
                          train  2925
                          val     361
           Forged_JPEG90  test    335
                          train  2925
                          val     361

표준 테이블 정확 일치 비율·progressive 비율


files exact_ratio  progressive_ratio
dataset    folder         split                                      
AutoSplice Authentic      test     228         1.0                0.0
                          train   1818         1.0                0.0
                          val      227         1.0                0.0
           Forged_JPEG100 test     335         1.0                0.0
                          train   2925         1.0                0.0
                          val      361         1.0                0.0
           Forged_JPEG75  test     335         1.0                0.0
                          train   2925         1.0                0.0
                          val      361         1.0                0.0
           Forged_JPEG90  test     335         1.0                0.0
                          train   2925         1.0                0.0
                          val      361         1.0                0.0

In [15]:
expected = {"Authentic": 75, "Forged_JPEG100": 100, "Forged_JPEG90": 90, "Forged_JPEG75": 75}
auto = jpeg[jpeg["dataset"] == "AutoSplice"].copy()
auto["expected_quality"] = auto["folder"].map(expected)
auto["matches_expected"] = (auto["quality"] == auto["expected_quality"]) & (auto["subsampling"] == 2) & auto["exact"]
print("기대 조건(정상 Q75, 조작본 폴더 이름의 품질, 4:2:0, 표준 테이블 정확 일치)과 다른 파일")
display(auto.groupby(["folder", "split"])["matches_expected"].agg(files="size", mismatched=lambda s: int((~s).sum())))
mismatch = auto[~auto["matches_expected"]]
if len(mismatch):
    display(mismatch[["folder", "split", "quality", "exact", "subsampling", "progressive", "path"]]
            .assign(path=lambda df: df["path"].map(lambda p: p.name)).head(50))

기대 조건(정상 Q75, 조작본 폴더 이름의 품질, 4:2:0, 표준 테이블 정확 일치)과 다른 파일


files  mismatched
folder         split                   
Authentic      test     228       False
               train   1818       False
               val      227       False
Forged_JPEG100 test     335       False
               train   2925       False
               val      361       False
Forged_JPEG75  test     335       False
               train   2925       False
               val      361       False
Forged_JPEG90  test     335       False
               train   2925       False
               val      361       False

## 6. 결과 정리

자동 검사 결과를 요약함. 정상·조작의 저장 품질 차이가 전체 split에서 확인되면, RGB·ELA 입력 모두에서 모델이 압축 차이를 지름길로 쓸 수 있는지 팀과 논의해야 함. 이 노트북은 학습·평가 설정을 바꾸지 않음.

In [16]:
def dominant(frame):
    counts = frame["quality"].value_counts()
    return f"Q{counts.index[0]} {counts.iloc[0]}/{len(frame)}" if len(counts) else "해당 없음"


summary = {}
for folder in ("Authentic", "Forged_JPEG100", "Forged_JPEG90", "Forged_JPEG75"):
    for split in ("train", "val", "test"):
        part = auto[(auto["folder"] == folder) & (auto["split"] == split)]
        summary[f"AutoSplice {folder} {split}"] = dominant(part)
coco = history[history["dataset"] == "CocoGlide"]
summary["CocoGlide 형식"] = coco["format"].value_counts().to_dict()
summary["기대 조건과 다른 AutoSplice 파일"] = int((~auto["matches_expected"]).sum())
for key, value in summary.items():
    print(f"{key}: {value}")

AutoSplice Authentic train: Q75 1818/1818
AutoSplice Authentic val: Q75 227/227
AutoSplice Authentic test: Q75 228/228
AutoSplice Forged_JPEG100 train: Q100 2925/2925
AutoSplice Forged_JPEG100 val: Q100 361/361
AutoSplice Forged_JPEG100 test: Q100 335/335
AutoSplice Forged_JPEG90 train: Q90 2925/2925
AutoSplice Forged_JPEG90 val: Q90 361/361
AutoSplice Forged_JPEG90 test: Q90 335/335
AutoSplice Forged_JPEG75 train: Q75 2925/2925
AutoSplice Forged_JPEG75 val: Q75 361/361
AutoSplice Forged_JPEG75 test: Q75 335/335
CocoGlide 형식: {'PNG': 1024}
기대 조건과 다른 AutoSplice 파일: 0


### 사람의 검토 메모

- 정상·조작 저장 품질 차이가 전체 split에서 같은지: 미확인
- 기대 조건과 다른 파일에 대한 판단: 미확인
- 팀 논의 필요 사항(지름길 가능성, 평가 조건 영향): 미정

[개별 보고서](../../docs/eda/EXP_seonghun_003_jpeg_history_check.md) · [byhyun 001 보고서](../../docs/eda/EXP_byhyun_001_preprocessing_check.md) · [TODO](../../TODO.md)